# 06 - IV Diagnostics

Hausman test, overidentification (if multiple instruments), weak instrument tests, and robustness to instrument definition.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

In [ ]:
from linearmodels.iv import IV2SLS
from linearmodels.panel import PanelOLS

In [ ]:
panel_iv = panel.dropna(subset=["leaid","year","pm25_annual_mean","smoke_days","test_score_mean"]).copy()
panel_iv["year"] = panel_iv["year"].astype(int)
for col in ["test_score_mean","pm25_annual_mean","smoke_days","smoke_days_heavy"]:
    if col in panel_iv.columns:
        panel_iv[f"{col}_dm"] = (
            panel_iv[col]
            - panel_iv.groupby("leaid")[col].transform("mean")
            - panel_iv.groupby("year")[col].transform("mean")
            + panel_iv[col].mean()
        )

## Hausman test: is PM2.5 endogenous?

In [ ]:
# If Hausman rejects, OLS is inconsistent and IV is preferred
# Implemented via augmented regression (Durbin-Wu-Hausman)
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm

# First stage residuals
X_fs = sm.add_constant(panel_iv["smoke_days_dm"])
fs   = OLS(panel_iv["pm25_annual_mean_dm"], X_fs).fit()
panel_iv["pm25_resid"] = fs.resid

# Augmented regression
X_aug = sm.add_constant(panel_iv[["pm25_annual_mean_dm","pm25_resid"]])
aug   = OLS(panel_iv["test_score_mean_dm"], X_aug).fit()

t_stat  = aug.tvalues["pm25_resid"]
p_value = aug.pvalues["pm25_resid"]
print(f"Hausman (DWH) test:")
print(f"  H0: PM2.5 is exogenous (OLS consistent)")
print(f"  t = {t_stat:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("  → Reject H0: endogeneity confirmed; use IV")
else:
    print("  → Fail to reject H0: OLS may be consistent")

## Alternative instrument: heavy smoke days only

In [ ]:
if "smoke_days_heavy_dm" in panel_iv.columns:
    iv_heavy = IV2SLS(
        dependent=panel_iv["test_score_mean_dm"],
        exog=None,
        endog=panel_iv[["pm25_annual_mean_dm"]],
        instruments=panel_iv[["smoke_days_heavy_dm"]],
    ).fit(cov_type="robust")
    b_heavy = iv_heavy.params["pm25_annual_mean_dm"]
    print(f"IV (heavy smoke only): β = {b_heavy:.4f}")
    print(f"IV (all smoke):        β = (see notebook 05)")
    print("Similar estimates = instrument definition doesn't drive results")
else:
    print("smoke_days_heavy not available in panel — check smoke instrument build")

## Robustness: drop 2018 Camp Fire year

In [ ]:
panel_nocamp = panel_iv[panel_iv["year"] != 2018].copy()
for col in ["test_score_mean","pm25_annual_mean","smoke_days"]:
    panel_nocamp[f"{col}_dm"] = (
        panel_nocamp[col]
        - panel_nocamp.groupby("leaid")[col].transform("mean")
        - panel_nocamp.groupby("year")[col].transform("mean")
        + panel_nocamp[col].mean()
    )
iv_nocamp = IV2SLS(
    dependent=panel_nocamp["test_score_mean_dm"],
    exog=None,
    endog=panel_nocamp[["pm25_annual_mean_dm"]],
    instruments=panel_nocamp[["smoke_days_dm"]],
).fit(cov_type="robust")
print(f"IV excl. 2018 (Camp Fire): β = {iv_nocamp.params['pm25_annual_mean_dm']:.4f}")
print("Similar to full-sample estimate = Camp Fire not driving results")